# `query_sector` — User Guide

Query the financial news pipeline by **sector** using the high-level Python API.
Sectors come from a fixed 19-value taxonomy — names are validated on every call.

**No API key required.** All functions read from `data/sector_summary.tsv`.

```
pipeline/query_sector.py
│
├── Discovery
│   └── list_sectors()                                     → list[str]
│
├── Point queries
│   ├── get_snapshot(sector)                               → dict
│   └── get_time_series(sector, lookback_days=30,
│                        include_articles=False)           → dict
│
└── Bulk export
    ├── get_all_sectors_pivot(lookback_days=None, freq="D") → pd.DataFrame
    └── export_sector_pivot(output_path=None,
                             lookback_days=90, freq="D")   → Path
```

| Function | Best for |
|---|---|
| `list_sectors` | discovering the 19 valid sector names |
| `get_snapshot` | "what is the current read on Technology Services?" |
| `get_time_series` | "how has Finance sentiment trended this quarter?" |
| `get_all_sectors_pivot` | wide date × sector matrix for heatmaps or correlation |
| `export_sector_pivot` | writing the pivot TSV for R or downstream tools |

## Sections
1. [Setup](#1-setup)
2. [Configuration](#2-configuration)
3. [Discover sectors](#3-discover-sectors--list_sectors)
4. [Sector snapshot](#4-sector-snapshot--get_snapshot)
5. [Sector time series](#5-sector-time-series--get_time_series)
6. [All-sectors pivot](#6-all-sectors-pivot--get_all_sectors_pivot)
7. [Export to TSV](#7-export-to-tsv--export_sector_pivot)
8. [End-to-end: sector heatmap](#8-end-to-end-sector-heatmap)
9. [Error handling reference](#9-error-handling-reference)

---
## 1 — Setup

Adds the project root to `sys.path` and pulls `sector_summary.tsv` from
HuggingFace if it doesn't exist locally.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.query_sector import (
    list_sectors,
    get_snapshot,
    get_time_series,
    get_all_sectors_pivot,
    export_sector_pivot,
)
from pipeline.constants import SECTOR_SUMMARY_FILE, SECTOR_PIVOT_FILE

print(f"sector_summary.tsv : {SECTOR_SUMMARY_FILE}")
print(f"pivot output       : {SECTOR_PIVOT_FILE}")

# Pull sector_summary.tsv from HuggingFace if it doesn't exist locally.
if not SECTOR_SUMMARY_FILE.exists():
    print("\nsector_summary.tsv not found — pulling from HuggingFace ...")
    from dotenv import load_dotenv
    from datasets import load_dataset

    load_dotenv(PROJECT_ROOT / ".env")
    hf_repo  = os.environ.get("HUGGINGFACE_REPO", "lacetohf/feeds")
    hf_user  = hf_repo.split("/")[0]
    repo_id  = f"{hf_user}/sector-analysis"

    ds = load_dataset(repo_id, split="train")
    df_hf = ds.to_pandas()

    SECTOR_SUMMARY_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_hf.to_csv(SECTOR_SUMMARY_FILE, sep="\t", index=False)
    print(f"Saved {len(df_hf):,} rows across {df_hf['date'].nunique()} dates -> {SECTOR_SUMMARY_FILE}")

print(f"\nFile exists : {SECTOR_SUMMARY_FILE.exists()}")

---
## 2 — Configuration

Change these values to explore different sectors and windows.
All cells below read from these variables.

In [ ]:
SECTOR   = "Technology Services"   # sector to query (must match taxonomy exactly)
LOOKBACK = 90                      # rolling window in days for time-series queries
FREQ     = "D"                     # resampling freq for pivot: "D" daily, "W" weekly, "M" monthly

---
## 3 — Discover sectors — `list_sectors`

Returns the full fixed taxonomy of 19 sector names. Unlike entities, this list
is static — it comes from `constants.SECTOR_TAXONOMY`, not from the data.
Every other function validates its `sector` argument against this list and
raises `ValueError` on a mismatch.

In [ ]:
sectors = list_sectors()

print(f"Total sectors : {len(sectors)}")
print()
for s in sectors:
    print(f"  {s}")

---
## 4 — Sector snapshot — `get_snapshot`

Returns the most recent single-day read for a sector: sentiment, score,
top entities mentioned that day, news category, and data age.

**When to use:** "What is the current read on Technology Services?"

**Return schema:**

| Key | Type | Description |
|---|---|---|
| `sector` | str | sector name |
| `last_date` | str | YYYY-MM-DD of the most recent entry |
| `latest_sentiment` | str | positive / neutral / negative |
| `sentiment_score` | int | −1 / 0 / +1 |
| `entities` | list[str] | named companies/orgs mentioned that day |
| `news_category` | str | earnings / M&A / regulation / macro / … |
| `extraction_status` | str | ok / partial |
| `data_age_days` | int | days since `last_date` |

**Key difference from `query_entity`:** sector names are exact-match against a fixed
taxonomy — no fuzzy lookup, no hints. Use `list_sectors()` first if unsure.

In [ ]:
snap = get_snapshot(SECTOR)

print(f"Sector             : {snap['sector']}")
print(f"Last date          : {snap['last_date']}")
print(f"Data age           : {snap['data_age_days']} days")
print(f"Latest sentiment   : {snap['latest_sentiment']}  (score={snap['sentiment_score']:+d})")
print(f"News category      : {snap['news_category']}")
print(f"Extraction status  : {snap['extraction_status']}")
print(f"Entities mentioned : {snap['entities']}")

In [ ]:
# Snapshot for every sector at once — useful for a quick market-wide overview.
print(f"{'sector':<28} {'sentiment':<12} {'score':>5}  {'age':>4}d  {'news_category'}")
print("-" * 72)
for s in list_sectors():
    try:
        snap_s = get_snapshot(s)
        print(f"{s:<28} {snap_s['latest_sentiment']:<12} {snap_s['sentiment_score']:>5}  "
              f"{snap_s['data_age_days']:>4}  {snap_s['news_category']}")
    except LookupError:
        print(f"{s:<28} {'(no data)':<12}")

---
## 5 — Sector time series — `get_time_series`

Returns sentiment trend, top entities, and category breakdown over a rolling
window. One row per calendar date the sector appears in the window.

**When to use:** "How has Finance sentiment trended over the last quarter?"

**Parameters:**

| Parameter | Default | Description |
|---|---|---|
| `sector` | required | exact taxonomy name |
| `lookback_days` | 30 | calendar days to look back from today |
| `include_articles` | False | also fetch raw headlines from feed files |

**Trend direction** threshold: ±0.20 (second-half mean − first-half mean).

In [ ]:
ts = get_time_series(SECTOR, lookback_days=LOOKBACK)

print(f"Sector             : {ts['sector']}")
print(f"Window             : {ts['date_range']['from']} -> {ts['date_range']['to']}")
print(f"Observations       : {ts['n_observations']} days")
print(f"Mean score         : {ts['mean_sentiment_score']:.3f}")
print(f"Trend              : {ts['trend_direction']}  (delta={ts['trend_delta']:+.3f})")
print(f"Dominant sentiment : {ts['dominant_sentiment']}")
print(f"Top entities       : {ts['top_entities'][:8]}")
print(f"\nCategory breakdown : {ts['category_breakdown']}")
print(f"\nLast 5 rows:")
print(f"  {'date':<12} {'sentiment':<12} {'score':>5}  {'news_category':<12}  entities")
print(f"  {'-'*70}")
for row in ts["time_series"][-5:]:
    ents = ", ".join(row["entities"][:3])
    print(f"  {row['date']:<12} {row['sentiment']:<12} {row['sentiment_score']:>5}  {row['news_category']:<12}  {ents}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

if ts["time_series"]:
    df_ts = pd.DataFrame(ts["time_series"])
    df_ts["date"] = pd.to_datetime(df_ts["date"])
    df_ts = df_ts.set_index("date").sort_index()

    rolling = df_ts["sentiment_score"].rolling(7, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#27ae60" if s > 0 else "#c0392b" if s < 0 else "#95a5a6"
              for s in df_ts["sentiment_score"]]
    ax.bar(df_ts.index, df_ts["sentiment_score"], color=colors, alpha=0.5, label="Daily score")
    ax.plot(df_ts.index, rolling, color="#2c3e50", linewidth=2, label="7-day rolling mean")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=45, ha="right")
    ax.set_ylabel("Sentiment score  (-1 / 0 / +1)")
    ax.set_title(f"{SECTOR} — {LOOKBACK}-day sentiment trend  |  {ts['trend_direction']}")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 6 — All-sectors pivot — `get_all_sectors_pivot`

Returns a **wide** `pd.DataFrame`: dates on the index, one column per sector,
values are mean daily sentiment scores. NaN = sector not seen on that date.

**When to use:** cross-sector correlation, heatmaps, or feeding into stats libs.

The `freq` parameter resamples rows:
- `"D"` — daily (default, one row per calendar day)
- `"W"` — weekly mean
- `"M"` — monthly mean

Columns are always alphabetically sorted — same order as `list_sectors()`.

In [ ]:
pivot = get_all_sectors_pivot(lookback_days=LOOKBACK, freq=FREQ)

print(f"Shape      : {pivot.shape}  (dates x sectors)")
print(f"Date range : {pivot.index.min().date()} -> {pivot.index.max().date()}")
print(f"NaN %      : {pivot.isna().mean().mean():.1%}  (days sector not seen)")
print(f"\nMean score per sector (last {LOOKBACK} days):")
means = pivot.mean().sort_values(ascending=False)
for sector, score in means.items():
    bar = "#" * int((score + 1) * 8)
    print(f"  {sector:<28} {score:>6.3f}  {bar}")

pivot.tail(3)

---
## 7 — Export to TSV — `export_sector_pivot`

Writes the wide pivot to a TSV file and returns the path.

- Default output: `data/sector_sentiment_pivot.tsv` (`SECTOR_PIVOT_FILE`)
- Default window: 90 days (`EXPORT_LOOKBACK_DAYS`)
- This is the same file pushed to `lacetohf/sector-analysis` by CI

Pass `freq="W"` or `freq="M"` to write a weekly/monthly aggregation instead.

In [ ]:
import tempfile

with tempfile.NamedTemporaryFile(suffix=".tsv", delete=False) as f:
    tmp_path = f.name

out_path = export_sector_pivot(output_path=tmp_path, lookback_days=LOOKBACK, freq=FREQ)

import pandas as pd
df_exported = pd.read_csv(out_path, sep="\t", index_col=0)

print(f"Written to : {out_path}")
print(f"Shape      : {df_exported.shape}")
print(f"Columns    : {list(df_exported.columns)[:5]} ... ({len(df_exported.columns)} total)")
df_exported.head(3)

---
## 8 — End-to-end: sector heatmap

Combines `get_all_sectors_pivot` with a diverging heatmap to give a
market-wide sentiment overview at a glance — all 19 sectors × last N days.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pivot_heat = get_all_sectors_pivot(lookback_days=LOOKBACK, freq=FREQ)

if pivot_heat.empty:
    print("No data available.")
else:
    # Sort sectors by mean score (most positive on left).
    col_order = pivot_heat.mean().sort_values(ascending=False).index.tolist()
    data = pivot_heat[col_order].values

    fig, ax = plt.subplots(figsize=(16, max(4, len(pivot_heat) * 0.18)))
    im = ax.imshow(data.T, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1,
                   interpolation="nearest")

    # X-axis: dates (every ~10th row label to avoid clutter).
    n_dates = len(pivot_heat)
    step = max(1, n_dates // 12)
    x_ticks = list(range(0, n_dates, step))
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(
        [str(pivot_heat.index[i].date()) for i in x_ticks],
        rotation=45, ha="right", fontsize=8,
    )

    # Y-axis: sector names.
    ax.set_yticks(range(len(col_order)))
    ax.set_yticklabels(col_order, fontsize=9)

    plt.colorbar(im, ax=ax, label="Mean sentiment score  (-1 negative / +1 positive)",
                 fraction=0.02, pad=0.02)
    ax.set_title(f"All-sector sentiment heatmap — last {LOOKBACK} days  (freq={FREQ})")
    plt.tight_layout()
    plt.show()

    # Print overall ranking.
    print(f"\nSector ranking by mean score (last {LOOKBACK} days):")
    for i, s in enumerate(col_order, 1):
        score = pivot_heat[s].mean()
        print(f"  {i:>2}. {s:<28} {score:>+.3f}")

---
## 9 — Error handling reference

### Guard conditions per function

| Function | Guard | Error |
|---|---|---|
| `get_snapshot` | sector name not in taxonomy | `ValueError` with full valid list |
| `get_snapshot` | sector in taxonomy but no data yet | `LookupError` |
| `get_time_series` | sector name not in taxonomy | `ValueError` |
| `get_time_series` | no data within `lookback_days` window | `LookupError` |
| `export_sector_pivot` | pivot empty (no data) | `RuntimeError` |

`list_sectors` and `get_all_sectors_pivot` **never raise** — they return a
static list and an empty DataFrame respectively.

**Key difference from `query_entity`:** bad sector names raise `ValueError`
(not `LookupError`) and the message lists every valid value — no fuzzy hints
because the taxonomy is fixed and small.

In [ ]:
# Bad sector name → ValueError with full valid list.
try:
    get_snapshot("Tech")
except ValueError as e:
    print(f"ValueError: {str(e)[:120]} ...")

In [ ]:
# Valid sector but no data in a very short window → LookupError.
try:
    get_time_series(SECTOR, lookback_days=1)
except LookupError as e:
    print(f"LookupError: {e}")

In [ ]:
# export_sector_pivot with a window too narrow → RuntimeError.
import tempfile, os

try:
    export_sector_pivot(
        output_path=os.path.join(tempfile.gettempdir(), "test_pivot.tsv"),
        lookback_days=1,
    )
except RuntimeError as e:
    print(f"RuntimeError: {e}")